# Correct Folding in 3D Slices â€” Multi-Constraint Comparison

Same synthetic 3D warp as `correct-3d-slices.ipynb`, but each slice is
corrected under four constraint modes and the results are compared using
the injectivity checks from `test-global-folding.ipynb`:

| Mode | `enforce_shoelace` | `enforce_injectivity` |
|------|-------------------|----------------------|
| jdet-only | False | False |
| jdet+shoe | True | False |
| jdet+inject | False | True |
| jdet+shoe+inject | True | True |

For each slice the notebook shows:
1. Jacobian-determinant heat-maps: before correction + after each mode (side-by-side)
2. Per-mode injectivity summary printed to stdout (Jdet, Shoelace, Monotonicity)
3. A final summary table across all slices and modes


## Imports

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import time

import numpy as np
import matplotlib.pyplot as plt

from dvfopt import (
    jacobian_det2D,
    iterative_serial,
    iterative_parallel,
    shoelace_det2D,
)
from dvfopt.jacobian import _monotonicity_diffs_2d
from dvfopt.jacobian.monotonicity import _diagonal_monotonicity_diffs_2d
from dvfopt.jacobian.intersection import has_quad_self_intersections
from dvfopt.viz import plot_grid_before_after
from benchmark_utils import (
    get_output_dir, save_figure, save_results_csv, save_summary_json, log_run_header, log_run_footer, results_to_rows, show_and_save, reset_figure_counter,
)


In [ ]:
METHOD = "slsqp"
NOTEBOOK_NAME = "3d-slices-multi-constraint"
OUTPUT_DIR = get_output_dir(METHOD, NOTEBOOK_NAME, base="../output")
reset_figure_counter()
summary = log_run_header(METHOD, NOTEBOOK_NAME, OUTPUT_DIR)


## Configuration

In [ ]:
# ── Solver ───────────────────────────────────────────────────────────────────
# Switch between iterative_serial (safer, single-process) and
# iterative_parallel (faster on multi-core, avoids Windows spawn overhead
# for single-window cases).
SOLVER = iterative_serial
# SOLVER = iterative_parallel

# ── Jdet colormap cap ─────────────────────────────────────────────────────
# Values beyond ±JDET_VMAX are clamped so extreme outliers don't wash out
# the colour scale around zero.
JDET_VMAX = 10

# ── Solver thresholds ────────────────────────────────────────────────────
JDET_THRESHOLD = 0.01

# Explicit injectivity_threshold bypasses the adaptive doubling loop, which
# would otherwise run the full solver up to (max_doublings+1) times.  0.3 is
# a good default for large-shear / radial-collapse fields.
INJECTIVITY_THRESHOLD = 0.3

MODES = [
    ("jdet-only",         dict(enforce_shoelace=False, enforce_injectivity=False)),
    ("jdet+shoe",         dict(enforce_shoelace=True,  enforce_injectivity=False)),
    ("jdet+inject",       dict(enforce_shoelace=False, enforce_injectivity=True,
                               injectivity_threshold=INJECTIVITY_THRESHOLD)),
    ("jdet+shoe+inject",  dict(enforce_shoelace=True,  enforce_injectivity=True,
                               injectivity_threshold=INJECTIVITY_THRESHOLD)),
]

SLICE_LABELS = [
    "crossing blobs", "tanh pinch", "sine fold",
    "radial implosion", "y-squeeze", "local push",
]


## Generate synthetic warp

In [ ]:
H, W, D = 20, 20, 6  # 40, 40, 6
yy, xx = np.mgrid[0:H, 0:W].astype(float)
cy, cx = H / 2.0, W / 2.0

warp_zyx = np.zeros((3, H, W, D), dtype=np.float64)

# Slice 0: Crossing blobs
ra = np.sqrt((yy - H*0.3)**2 + (xx - W*0.3)**2)
rb = np.sqrt((yy - H*0.7)**2 + (xx - W*0.7)**2)
ma = (ra < H*0.15).astype(float)
mb = (rb < H*0.15).astype(float)
warp_zyx[1, :, :, 0] = ma * H*0.4 - mb * H*0.4
warp_zyx[2, :, :, 0] = ma * W*0.4 - mb * W*0.4

# Slice 1: Tanh pinch
warp_zyx[1, :, :, 1] = -(H * 0.15) * np.tanh(0.3 * (yy - cy))

# Slice 2: Sine fold
warp_zyx[2, :, :, 2] = (W * 0.2) * np.sin(4 * np.pi * xx / W)

# Slice 3: Radial implosion
r3 = np.sqrt((yy - cy)**2 + (xx - cx)**2) + 1e-6
angle3 = np.arctan2(yy - cy, xx - cx)
strength3 = np.exp(-r3**2 / (2 * (H*0.2)**2)) * H * 0.7
warp_zyx[1, :, :, 3] = -strength3 * np.sin(angle3)
warp_zyx[2, :, :, 3] = -strength3 * np.cos(angle3)

# Slice 4: Y-squeeze
warp_zyx[1, :, :, 4] = (
    np.where(yy < cy, H*0.25, -H*0.25)
    * np.exp(-((yy - cy) / (H*0.15))**2)
)

# Slice 5: Local push
r5 = np.sqrt((yy - H*0.4)**2 + (xx - W*0.4)**2) + 1e-6
strength5 = np.exp(-r5**2 / (2*(H*0.08)**2)) * H * 0.8
warp_zyx[1, :, :, 5] = strength5 * (yy - H*0.4) / r5
warp_zyx[2, :, :, 5] = strength5 * (xx - W*0.4) / r5


print(f"Generated synthetic warp: (3, {H}, {W}, {D})")
for z in range(D):
    dy_s = warp_zyx[1, :, :, z]
    dx_s = warp_zyx[2, :, :, z]
    jac_z = jacobian_det2D(np.stack([dy_s, dx_s]))
    n_neg = int((jac_z <= 0).sum())
    print(f"  slice {z} ({SLICE_LABELS[z]}):  neg_jdet={n_neg}  min={float(jac_z.min()):+.4f}")


## Deformation field convention

Displacement fields are **pull-back (backward) maps**: for every voxel
position in **fixed space** the warp gives the displacement to the corresponding
location in **moving space**.  This is identical to the dvfopt convention
(`[dz, dy, dx]` in fixed-space coordinates, pull-back semantics).

The synthetic slices below follow the same convention: each slice shows how
fixed-space grid nodes are displaced when sampling the moving image.


In [ ]:
# Show the planned warping for each slice as a deformed grid
# (pull-back: pull-back displacement field: for each fixed-space voxel, offset to the sample location)
stride = 2

fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)

# Track the last pcolormesh for the shared colorbar
pcm = None

for z, ax in enumerate(axes.flat):
    dy = warp_zyx[1, :, :, z]
    dx = warp_zyx[2, :, :, z]

    phi_z = np.stack([dy, dx])
    jac_z = np.squeeze(jacobian_det2D(phi_z))
    n_neg = int((jac_z <= 0).sum())
    vmax = min(max(abs(float(jac_z.min())), float(jac_z.max()), 1.0), JDET_VMAX)

    # Build strided vertex indices — identical for pcolormesh AND grid lines
    rows_s = np.arange(0, H, stride)
    cols_s = np.arange(0, W, stride)
    if rows_s[-1] != H - 1:
        rows_s = np.append(rows_s, H - 1)
    if cols_s[-1] != W - 1:
        cols_s = np.append(cols_s, W - 1)

    rr, cc = np.meshgrid(rows_s, cols_s, indexing="ij")
    Y_v = rr + dy[rr, cc]
    X_v = cc + dx[rr, cc]

    jac_sub = jac_z[np.ix_(rows_s[:-1], cols_s[:-1])]

    pcm = ax.pcolormesh(X_v, Y_v, jac_sub,
                        cmap="RdBu_r", vmin=-vmax, vmax=vmax, alpha=0.55, shading="flat")

    for i in range(Y_v.shape[0]):
        ax.plot(X_v[i, :], Y_v[i, :], "k-", lw=0.7, alpha=0.85)
    for j in range(Y_v.shape[1]):
        ax.plot(X_v[:, j], Y_v[:, j], "k-", lw=0.7, alpha=0.85)

    pad = 0.5
    ax.set_xlim(float(X_v.min()) - pad, float(X_v.max()) + pad)
    ax.set_ylim(float(Y_v.max()) + pad, float(Y_v.min()) - pad)
    ax.set_aspect("equal")
    ax.axis("off")

    subtitle = f"neg_jdet={n_neg}  min={float(jac_z.min()):+.3f}"
    ax.set_title(f"Slice {z}: {SLICE_LABELS[z]}" + chr(10) + subtitle, fontsize=9)

fig.colorbar(pcm, ax=axes.ravel().tolist(), label="Jacobian determinant", shrink=0.6, pad=0.02)

plt.suptitle(
    "Planned warping -- deformed grid per slice  |  pull-back displacement field",
    fontsize=11,
)
show_and_save(OUTPUT_DIR)


## Helpers

In [ ]:
def extract_phi(warp_zyx, z):
    dy = warp_zyx[1, :, :, z].copy()
    dx = warp_zyx[2, :, :, z].copy()
    deformation = np.zeros((3, 1, H, W), dtype=np.float64)
    deformation[1, 0] = dy
    deformation[2, 0] = dx
    return deformation


def injectivity_stats(phi):
    jac  = np.squeeze(jacobian_det2D(phi))
    shoe = np.squeeze(shoelace_det2D(phi))
    h_m, v_m = _monotonicity_diffs_2d(phi[0], phi[1])
    d1, d2   = _diagonal_monotonicity_diffs_2d(phi[0], phi[1])
    # Global check: any non-adjacent quad cells intersect geometrically?
    # O(n^2) — slow on large grids but definitive.
    global_intersect = has_quad_self_intersections(phi)
    return dict(
        n_neg_jac        = int((jac[1:-1, 1:-1] <= 0).sum()),
        n_neg_shoe       = int((shoe[1:-1, 1:-1] <= 0).sum()),
        n_h_viol         = int((h_m[1:-1, 1:-1] <= 0).sum()),
        n_v_viol         = int((v_m[1:-1, 1:-1] <= 0).sum()),
        n_d1_viol        = int((d1[1:-1, 1:-1] <= 0).sum()),
        n_d2_viol        = int((d2[1:-1, 1:-1] <= 0).sum()),
        global_intersect = global_intersect,
        min_jac          = float(jac.min()),
        jac              = jac,
    )


def show_jdet_map(jac2d, title, ax):
    raw_vmax = max(abs(float(jac2d.min())), abs(float(jac2d.max())), 1.0)
    vmax = min(raw_vmax, JDET_VMAX)
    im = ax.imshow(jac2d, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    n_neg = int((jac2d <= 0).sum())
    ax.set_title(title + chr(10) + f"neg={n_neg}  min={jac2d.min():+.3f}", fontsize=8)
    ax.axis("off")
    return im


## Run corrections across all modes

In [ ]:
all_results = {}   # {z: {mode_label: stats_dict}}

for z in range(D):
    label = SLICE_LABELS[z]
    deformation = extract_phi(warp_zyx, z)
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])

    stats_init = injectivity_stats(phi_init)
    n_neg_init = stats_init['n_neg_jac']

    sep = "=" * 80
    print()
    print(sep)
    print(f"  Slice {z}  ({label})  |  {H}x{W}  |  neg_jdet={n_neg_init}  min={stats_init['min_jac']:+.4f}")
    print(sep)

    mode_results = {}

    for mode_label, flags in MODES:
        t0 = time.perf_counter()
        phi_c = SOLVER(
            deformation.copy(), verbose=0, threshold=JDET_THRESHOLD, **flags
        )
        elapsed = time.perf_counter() - t0

        st = injectivity_stats(phi_c)
        st['time'] = elapsed
        st['phi']  = phi_c
        mode_results[mode_label] = st

        ok_j  = "OK" if st['n_neg_jac']        == 0     else f"FAIL({st['n_neg_jac']})"
        ok_s  = "OK" if st['n_neg_shoe']       == 0     else f"FAIL({st['n_neg_shoe']})"
        ok_h  = "OK" if st['n_h_viol']         == 0     else f"FAIL({st['n_h_viol']})"
        ok_v  = "OK" if st['n_v_viol']         == 0     else f"FAIL({st['n_v_viol']})"
        ok_d1 = "OK" if st['n_d1_viol']        == 0     else f"FAIL({st['n_d1_viol']})"
        ok_d2 = "OK" if st['n_d2_viol']        == 0     else f"FAIL({st['n_d2_viol']})"
        ok_g  = "OK" if not st['global_intersect']      else "INTERSECT"
        print(f"  [{mode_label:20s}]  jdet={ok_j:12s}  shoe={ok_s:12s}  "
              f"h={ok_h:10s}  v={ok_v:10s}  d1={ok_d1:10s}  d2={ok_d2:10s}  "
              f"glob={ok_g:9s}  t={elapsed:.2f}s")

        plot_grid_before_after(
            deformation,
            phi_c,
            title=f"Slice {z} ({label}) — {mode_label}",
            jdet_vmax=JDET_VMAX,
        )

    # --- Jdet heatmap summary: before + all modes side-by-side ---
    ncols = 1 + len(MODES)
    fig, axes = plt.subplots(1, ncols, figsize=(3.5 * ncols, 3.2), constrained_layout=True)
    im_ref = show_jdet_map(stats_init['jac'], "Before", axes[0])
    for col, (mode_label, _) in enumerate(MODES, start=1):
        show_jdet_map(mode_results[mode_label]['jac'], mode_label, axes[col])
    plt.colorbar(im_ref, ax=axes.tolist(), shrink=0.7, label="Jacobian det")
    plt.suptitle(f"Slice {z} ({label}) — Jdet summary across all modes", fontsize=10)
    show_and_save(OUTPUT_DIR)

    all_results[z] = mode_results


## Summary table

In [ ]:
def tick(n):
    return "  OK  " if n == 0 else f"FAIL({n})"

hdr = (f"{'Sl':>2s}  {'Pattern':<18s}  {'Mode':<20s}  "
       f"{'Jdet':>10s}  {'Shoelace':>10s}  "
       f"{'Mono-h':>8s}  {'Mono-v':>8s}  {'Mono-d1':>9s}  {'Mono-d2':>9s}  "
       f"{'Glob-X':>9s}  {'Time':>6s}")
print(hdr)
print("-" * len(hdr))

for z in range(D):
    for i, (mode_label, _) in enumerate(MODES):
        st = all_results[z][mode_label]
        pat = SLICE_LABELS[z] if i == 0 else ""
        sl  = str(z) if i == 0 else ""
        g   = "INTERSECT" if st['global_intersect'] else "  OK"
        print(f"{sl:>2s}  {pat:<18s}  {mode_label:<20s}  "
              f"{tick(st['n_neg_jac']):>10s}  {tick(st['n_neg_shoe']):>10s}  "
              f"{tick(st['n_h_viol']):>8s}  {tick(st['n_v_viol']):>8s}  "
              f"{tick(st['n_d1_viol']):>9s}  {tick(st['n_d2_viol']):>9s}  "
              f"{g:>9s}  {st['time']:>5.2f}s")
    print()


In [ ]:
# --- Save results ---
if 'results' in dir() and isinstance(results, dict) and results:
    rows, cols = results_to_rows(results)
    save_results_csv(rows, cols, OUTPUT_DIR)
    summary = log_run_footer(summary, results)
    save_summary_json(summary, OUTPUT_DIR)
elif 'mag_results' in dir():
    rows, cols = results_to_rows(mag_results)
    save_results_csv(rows, cols, OUTPUT_DIR, name='results_magnitude')
    if 'density_results' in dir():
        rows2, cols2 = results_to_rows(density_results)
        save_results_csv(rows2, cols2, OUTPUT_DIR, name='results_density')
    combined = {**mag_results, **density_results} if 'density_results' in dir() else mag_results
    summary = log_run_footer(summary, combined)
    save_summary_json(summary, OUTPUT_DIR)
else:
    save_summary_json(summary, OUTPUT_DIR)
    print('No results dict found; saved summary only.')
